## Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

print("Bibliotheques importees.")

Bibliotheques importees.


## Parametres

In [2]:
INPUT_FILE   = "../../data/concrete/concrete_data.csv"
TARGET_COL   = "Concrete compressive strength"
RANDOM_STATE = 42
TEST_SIZE    = 0.2

OUTPUT_DIR = "../../outputs/concrete"

## Chargement et nettoyage des données

In [3]:
# Ce fichier est deja propre (numerique, sans colonnes categorielles a retirer).
# On verifie tout de meme l'absence de valeurs manquantes par securite.
print("[SVM] Chargement du fichier source...")
data = pd.read_csv(INPUT_FILE, sep=";")

nb_na = data.isna().sum().sum()
print(f"[SVM] Valeurs manquantes detectees : {nb_na}")
data = data.dropna().reset_index(drop=True)

print(f"[SVM] Dimensions apres verification : {data.shape}")
data.head()

[SVM] Chargement du fichier source...
[SVM] Valeurs manquantes detectees : 0
[SVM] Dimensions apres verification : (1030, 9)


,Cement,Blast Furnac Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Concrete compressive strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


## Séparation train/test (80/20)

In [4]:
# IMPORTANT : aucune instance n'est mise de cote a ce stade. Ce meme split
# servira a la Phase 1 (motifs sur X_train uniquement) et a la Phase 3
# (contrefactuels generes sur les instances de X_test).
X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"[SVM] Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False, sep=";")
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False, sep=";")
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False, sep=";")
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False, sep=";")
print("[SVM] Splits sauvegardes dans outputs/concrete/")

[SVM] Train : 824 lignes | Test : 206 lignes
[SVM] Splits sauvegardes dans outputs/concrete/


## Entraînement du SVM

In [5]:
# Standardisation des features (pipeline) et de la cible (TransformedTargetRegressor).
# model.predict(x) travaille ensuite en entree/sortie sur les valeurs REELLES :
# les Phases 1 et 3 n'ont pas a se soucier de cette mise a l'echelle.
print("[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...")

base_pipeline = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
model = TransformedTargetRegressor(regressor=base_pipeline, transformer=StandardScaler())
model.fit(X_train, y_train)

print("[SVM] Entrainement termine.")

[SVM] Entrainement du modele SVR (noyau RBF, hyperparametres par defaut)...
[SVM] Entrainement termine.


## Évaluation

In [6]:
print("[SVM] Evaluation sur l'ensemble de test (holdout)...")

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"[SVM] RMSE : {rmse:.2f}")
print(f"[SVM] MAE  : {mae:.2f}")
print(f"[SVM] R2   : {r2:.4f}")
print(f"[SVM] MAPE : {mape:.2f}%")

pd.DataFrame([{"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}]).to_csv(
    f"{OUTPUT_DIR}/metriques_svm.csv", index=False, sep=";"
)

[SVM] Evaluation sur l'ensemble de test (holdout)...
[SVM] RMSE : 6.63
[SVM] MAE  : 4.89
[SVM] R2   : 0.8296
[SVM] MAPE : 16.18%


## Sauvegarde du modèle

In [7]:
joblib.dump(model, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"[SVM] Modele sauvegarde dans {OUTPUT_DIR}/svm_model.pkl")
print("[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.")

[SVM] Modele sauvegarde dans ../../outputs/concrete/svm_model.pkl
[SVM] Ce notebook n'a besoin d'etre execute qu'UNE SEULE FOIS.
